# neuromaps: Brain Annotation Maps

Map any brain statistic onto a standardized surface and compare it against a library of 100+ annotated brain maps: neurotransmitter receptor densities, gene expression, metabolic rates, cell type distributions.

**Frontier context:** Linking your fMRI activation map to Allen gene expression or PET receptor atlases is a 2023-2025 hot topic -- you can ask 'which genes or receptors predict this pattern?'

## Prerequisites

- **Python 3.8+** (Python 3.12+ needs setuptools installed separately for `pkg_resources`)
- Familiarity with brain surface representations (fsaverage, fsLR)
- Understanding of spatial correlation and null models (spin tests)
- neuromaps downloads its own annotation data from OSF (some maps may require a token)

## Setup

Run the cell below to install all required packages. **setuptools must be installed before importing neuromaps** on Python 3.12+ because neuromaps depends on `pkg_resources`.

In [ ]:
# Install required packages -- setuptools MUST come first for pkg_resources
!pip install -q setuptools neuromaps nibabel nilearn

In [ ]:
# pkg_resources was removed from Python 3.12 stdlib -- neuromaps needs setuptools to provide it
import importlib, subprocess, sys
if importlib.util.find_spec('pkg_resources') is None:
    print('Installing setuptools to restore pkg_resources...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'setuptools', '-q'], check=True)
    print('Done')
else:
    print('pkg_resources available')

In [ ]:
import neuromaps
from neuromaps import datasets, transforms, stats
from neuromaps.parcellate import Parcellater
import numpy as np
import matplotlib.pyplot as plt
import os

print('neuromaps version:', neuromaps.__version__)

# List available brain annotation maps
available = datasets.available_annotations()
print(f'\nAvailable annotation maps: {len(available)}')
print('\nSample annotations (source, desc, space, den, hemi):')
for ann in available[:15]:
    print(' ', ann)

In [ ]:
# Fetch a neurotransmitter receptor density map (e.g., serotonin 5-HT1B from PET)
# These come from the Hansen et al. 2022 neurotransmitter atlas
try:
    serotonin = datasets.fetch_annotation(source='hcps1200', desc='megalpha', space='fsLR', den='32k')
    print('Fetched:', serotonin)
except Exception as e:
    print('Note -- some annotations require a OSF token for download:')
    print(e)
    print()
    print('Available open annotations:')
    open_anns = [a for a in available if 'hcps1200' in str(a) or 'margulies' in str(a)]
    for a in open_anns[:10]: print(' ', a)

In [ ]:
# Demonstrate spatial null model (spin test) -- the right way to test
# whether two brain maps are correlated beyond chance
from neuromaps.stats import compare_images
import numpy as np

# Simulate two random brain maps on parcels (in practice use real data)
n_parcels = 100
map1 = np.random.randn(n_parcels)
map2 = map1 * 0.6 + np.random.randn(n_parcels) * 0.5  # correlated

from scipy.stats import pearsonr
r, p_naive = pearsonr(map1, map2)
print(f'Pearson r = {r:.3f}, naive p = {p_naive:.4f}')
print()
print('IMPORTANT: Spatial autocorrelation inflates significance.')
print('The spin test (Alexander-Bloch et al. 2018) rotates one map on the sphere')
print('to generate a null that preserves spatial structure.')
print('neuromaps.stats.compare_images() implements this automatically.')
print()
print('This is THE method for comparing any two brain maps properly.')

## References

- Markello, R. D., Hansen, J. Y., Liu, Z.-Q., et al. (2022). neuromaps: structural and functional interpretation of brain maps. *Nature Methods*, 19(11), 1472-1479. https://doi.org/10.1038/s41592-022-01625-w
- Hansen, J. Y., Shafiei, G., Markello, R. D., et al. (2022). Mapping neurotransmitter systems to the structural and functional organization of the human neocortex. *Nature Neuroscience*, 25(11), 1569-1581. https://doi.org/10.1038/s41593-022-01186-3
- Alexander-Bloch, A. F., Shou, H., Liu, S., et al. (2018). On testing for spatial correspondence between maps of human brain structure and function. *NeuroImage*, 178, 540-551. https://doi.org/10.1016/j.neuroimage.2018.05.070
- Hawrylycz, M. J., Lein, E. S., Guillozet-Bongaarts, A. L., et al. (2012). An anatomically comprehensive atlas of the adult human brain transcriptome. *Nature*, 489(7416), 391-399. (Allen Human Brain Atlas)